# Stage 03 — Embedding and vector store

**Track A (Buse) · Stage 3 of 10**

| | |
|---|---|
| **Input** | `data/processed/chunks.jsonl` |
| **Output** | A persisted Chroma collection, `papers_v1` |
| **Promotes to** | `src/research_assistant/ingestion/embed_B.py`, `src/research_assistant/retrieval/vector_store_B.py` |
| **Config** | `configs/ingestion_B.yaml` → `embed:`, `configs/retrieval_B.yaml` → `vector_store:` |

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Embedding model | `BAAI/bge-small-en-v1.5` | `bge-base`, `e5-base-v2`, `all-MiniLM-L6-v2` | 384 dimensions and 33M parameters. Runs on your CPU in minutes, and strong for its size. Start small so you can re-embed the whole corpus after every chunking change. |
| Asymmetric prefixes | Yes, query-side only | Symmetric | This model family is trained with a query instruction. Dropping it costs real accuracy, and it is the single most common silent misuse of these models. |
| Normalisation | Yes | Raw vectors | Cosine similarity requires it, and Chroma's cosine space assumes it. |
| Vector store | Chroma | Qdrant, FAISS, pgvector | Local, file-backed, zero services. Qdrant is the better production answer and has native hybrid, but it needs Docker, which is Sude's surface, not yours. The store sits behind `vector_store_B.py` so swapping it later is one file. |
| Collection naming | Versioned, `papers_v1` | Overwrite in place | Re-ingesting in place destroys your ability to compare a new chunking against the old index. Bump the version, keep both, delete when the gate has spoken. |

**Re-embedding is cheap at this corpus size, so treat the index as disposable.**
The moment you cannot rebuild it from `data/raw` plus configs, you have a
reproducibility problem your report cannot survive.

In [ ]:
from _nbsetup_B import REPO, DATA, load_cfg, resolve, ensure_dirs
import json
from pathlib import Path

icfg = load_cfg("ingestion")
rcfg = load_cfg("retrieval")
chunks = [json.loads(l) for l in (resolve(icfg["corpus"]["processed_dir"]) / "chunks.jsonl")
          .read_text(encoding="utf-8").splitlines() if l.strip()]
print(len(chunks), "chunks")

In [ ]:
from sentence_transformers import SentenceTransformer

e = icfg["embed"]
model = SentenceTransformer(e["model"], device=None if e["device"] == "auto" else e["device"])
print("dim:", model.get_sentence_embedding_dimension(), "expected:", e["dim"])
assert model.get_sentence_embedding_dimension() == e["dim"], "update embed.dim in the config"

def embed_passages(texts):
    return model.encode([e["passage_prefix"] + t for t in texts],
                        batch_size=e["batch_size"], normalize_embeddings=e["normalize"],
                        show_progress_bar=True)

def embed_query(q):
    return model.encode([e["query_prefix"] + q], normalize_embeddings=e["normalize"])[0]

In [ ]:
import chromadb

vs = rcfg["vector_store"]
client = chromadb.PersistentClient(path=str(resolve(vs["persist_dir"])))
coll = client.get_or_create_collection(vs["collection"], metadata={"hnsw:space": vs["distance"]})

vecs = embed_passages([c["text"] for c in chunks])
coll.upsert(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=[v.tolist() for v in vecs],
    documents=[c["text"] for c in chunks],
    metadatas=[{k: c[k] for k in ("paper_id", "title", "section", "page_start", "page_end")}
               for c in chunks],
)
print("collection size:", coll.count())

### Smoke test the index before trusting it

Query it with something you already know the answer to. If the top hit is obviously
wrong here, no amount of reranking in stage 08 will save you.

In [ ]:
q = "TODO: a question you already know the answer to"
res = coll.query(query_embeddings=[embed_query(q).tolist()], n_results=5)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"[{dist:.3f}] {meta['title'][:50]} / {meta['section'][:40]} p{meta['page_start']}")
    print("   ", doc[:180].replace(chr(10), " "), "\n")

### Model comparison

Swap the model, re-embed, re-measure in notebook 06. Log each run to MLflow so the
comparison survives the session. Fill the delta column into the tactic ledger.

| Model | Dim | Params | Notes |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 22M | Fastest, symmetric, weakest. A useful floor. |
| `bge-small-en-v1.5` | 384 | 33M | Current pick. |
| `bge-base-en-v1.5` | 768 | 109M | Usually a real gain, doubles index size and embed time. |
| `e5-base-v2` | 768 | 109M | Different prefix convention (`query:` / `passage:`). Getting the prefix wrong will look like the model being bad. |

## Exit checks

- [ ] Collection count equals chunk count.
- [ ] The smoke-test query returns something you would accept as an answer.
- [ ] The query prefix is applied on queries and not on passages. Check it, do not assume.
- [ ] The whole index rebuilds from scratch in one run of this notebook.

## Promote to `src/`

`embed_B.py` gets the two embed functions. `vector_store_B.py` gets a thin class with
`upsert` and `search`, so Qdrant can replace Chroma without anything upstream noticing.